In [2]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report,average_precision_score
import optuna
from optuna.trial import Trial
from scipy.stats import ks_2samp


import XGBOOST_module

In [3]:
data = pd.read_parquet("/Users/prathikkundaragi/MY PROJECTS/Fraud Dataset/model_data_full.parquet")
data.head(5)

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_OUT,type_TRANSFER
2,181.00,181.0,0.0,0.0,0.00,1,0,1
3,181.00,181.0,0.0,21182.0,0.00,1,1,0
15,229133.94,15325.0,0.0,5083.0,51513.44,0,1,0
19,215310.30,705.0,0.0,22425.0,0.00,0,0,1
24,311685.89,10835.0,0.0,6267.0,2719172.89,0,0,1


In [4]:

data_cash_out = data[data['type_CASH_OUT'] == 1].drop(columns=['type_CASH_OUT','type_TRANSFER'])

data_transfer = data[data['type_TRANSFER'] == 1].drop(columns=['type_CASH_OUT','type_TRANSFER'])

## CASH OUT PAYMENTS

In [5]:
#for data_cash_out
#creating train and holdout datasets
data_cash_out_train, data_cash_out_holdout = train_test_split(data_cash_out, test_size=0.3, random_state=42)

In [6]:
#creating X and y exogenous and endogenous variables for training
X_cash_out_train = data_cash_out_train.drop('isFraud', axis=1)
y_cash_out_train = data_cash_out_train['isFraud']


In [7]:
y_cash_out_train

2110104    0
4894554    0
5118477    0
442534     0
118630     0
          ..
2055620    0
313312     0
4776175    0
6336267    0
6307558    0
Name: isFraud, Length: 1566250, dtype: int64

In [8]:
#importing the XGBOOST module and running the tuning function for the full dataset

XGBOOST_cash_out= XGBOOST_module.run_fraud_tuning(X_cash_out_train, y_cash_out_train, n_trials=4)

  0%|          | 0/4 [00:00<?, ?it/s]

  CV   AUC-PR  (best trial) : 0.6677
  CV   KS Stat (best trial) : 0.9392
  Internal Test AUC-PR      : 0.6527
  Internal Test KS Stat     : 0.9326
  Best Params               : {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.11536162338241387, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5090949803242604, 'min_child_weight': 2, 'gamma': 1.5212112147976886, 'reg_alpha': 0.042051564509138696, 'reg_lambda': 0.01444525102276306, 'scale_pos_weight': 50.772079627725866}
  Final model retrained on full X_train.
  Apply to holdout externally in your prediction notebook.


In [9]:
#creating holdout X and y exogenous and endogenous variables for testing
X_holdout_cash_out = data_cash_out_holdout.drop('isFraud', axis=1)
y_holdout_cash_out = data_cash_out_holdout['isFraud']

In [10]:
fraud_score_cash_out = pd.DataFrame()
fraud_score_cash_out.index= X_holdout_cash_out.index
fraud_score_cash_out['fraud_score'] =  XGBOOST_cash_out['model'].predict_proba(X_holdout_cash_out)[:, 1]
fraud_score_cash_out['isFraud'] = y_holdout_cash_out
fraud_score_cash_out['amount'] = X_holdout_cash_out['amount']
fraud_score_cash_out.head(5)

,fraud_score,isFraud,amount
4076553,0.002670,0,48612.58
3236274,0.003862,0,82271.46
1477975,0.000042,0,11585.06
6085979,0.044183,0,90920.85
1218962,0.001473,0,156459.19


In [11]:
fraud_score_cash_out.sort_values(by='fraud_score', ascending=False, inplace=True)
fraud_score_cash_out.head(10)

,fraud_score,isFraud,amount
10396,0.999057,1,5460002.91
6168504,0.999057,1,9421856.87
6036620,0.999057,1,9639524.70
650134,0.999057,1,9465988.82
6362577,0.998847,1,10000000.00
3760285,0.998847,1,10000000.00
1059699,0.998847,1,10000000.00
6281481,0.998847,1,10000000.00
586312,0.998847,1,10000000.00
6009596,0.998847,1,10000000.00


In [12]:
# Create fraud_amount column — zero for non-fraud rows
fraud_score_cash_out['fraud_amount'] = fraud_score_cash_out['amount'] * fraud_score_cash_out['isFraud']

# Create score buckets
fraud_score_cash_out['score_bucket'] = pd.cut(fraud_score_cash_out['fraud_score'], 
                                      bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                                      labels=['0-0.1', '0.1-0.2', '0.2-0.3', '0.3-0.4', '0.4-0.5', 
                                             '0.5-0.6', '0.6-0.7', '0.7-0.8', '0.8-0.9', '0.9-1.0'])

# Build summary dataframe
fraud_by_bucket = fraud_score_cash_out.groupby('score_bucket', observed=True).agg(
    fraud_count  = ('isFraud',      'sum'),
    fraud_volume = ('fraud_amount', 'sum')
).reset_index()

# % of total fraud count and volume
fraud_by_bucket['pct_fraud_count']  = (fraud_by_bucket['fraud_count']  / fraud_by_bucket['fraud_count'].sum()  * 100).round(2)
fraud_by_bucket['pct_fraud_volume'] = (fraud_by_bucket['fraud_volume'] / fraud_by_bucket['fraud_volume'].sum() * 100).round(2)

fraud_by_bucket.sort_values(by='score_bucket', inplace=True)
fraud_by_bucket

,score_bucket,fraud_count,fraud_volume,pct_fraud_count,pct_fraud_volume
0,0-0.1,61,4.023539e+06,4.88,0.22
1,0.1-0.2,68,5.897395e+06,5.44,0.32
2,0.2-0.3,70,8.121363e+06,5.60,0.44
3,0.3-0.4,64,9.634420e+06,5.12,0.52
4,0.4-0.5,66,1.016707e+07,5.28,0.55
5,0.5-0.6,80,1.594178e+07,6.40,0.86
6,0.6-0.7,70,1.545293e+07,5.60,0.83
7,0.7-0.8,91,2.096080e+07,7.28,1.13
8,0.8-0.9,111,3.471441e+07,8.88,1.87
9,0.9-1.0,569,1.730464e+09,45.52,93.27


# TRASFER

In [13]:

#creating train and holdout datasets for transfer payments
data_transfer_train, data_transfer_holdout = train_test_split(data_transfer, test_size=0.3, random_state=42)


#creating X and y exogenous and endogenous variables for training
X_transfer_train = data_transfer_train.drop('isFraud', axis=1)
y_transfer_train = data_transfer_train['isFraud']

In [14]:
#importing the XGBOOST module and running the tuning function for the full dataset

XGBOOST_transfer= XGBOOST_module.run_fraud_tuning(X_transfer_train, y_transfer_train, n_trials=4)

  0%|          | 0/4 [00:00<?, ?it/s]

  CV   AUC-PR  (best trial) : 0.9952
  CV   KS Stat (best trial) : 0.9965
  Internal Test AUC-PR      : 0.9955
  Internal Test KS Stat     : 0.9965
  Best Params               : {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.11536162338241387, 'subsample': 0.6061695553391381, 'colsample_bytree': 0.5090949803242604, 'min_child_weight': 2, 'gamma': 1.5212112147976886, 'reg_alpha': 0.042051564509138696, 'reg_lambda': 0.01444525102276306, 'scale_pos_weight': 50.772079627725866}
  Final model retrained on full X_train.
  Apply to holdout externally in your prediction notebook.


In [15]:
#creating holdout X and y exogenous and endogenous variables for testing
X_holdout_transfer = data_transfer_holdout.drop('isFraud', axis=1)
y_holdout_transfer = data_transfer_holdout['isFraud']

In [16]:
fraud_score_transfer= pd.DataFrame()
fraud_score_transfer.index= X_holdout_transfer.index
fraud_score_transfer['fraud_score'] =  XGBOOST_transfer['model'].predict_proba(X_holdout_transfer)[:, 1]
fraud_score_transfer['isFraud'] = y_holdout_transfer
fraud_score_transfer['amount'] = X_holdout_transfer['amount']
fraud_score_transfer.head(5)

,fraud_score,isFraud,amount
4684385,0.000053,0,46083.92
2927383,0.000014,0,729344.84
4608378,0.000032,0,252234.22
2317293,0.000055,0,441247.78
1497357,0.000024,0,161790.31


In [17]:
fraud_score_transfer.sort_values(by='fraud_score', ascending=False, inplace=True)
fraud_score_transfer.head(10)

,fraud_score,isFraud,amount
6281500,0.999813,1,1086227.44
1059530,0.999813,1,1076739.91
5563759,0.999813,1,1084144.37
5186063,0.999813,1,1082308.05
4557165,0.999813,1,1076518.28
1030687,0.999781,1,10000000.00
6362460,0.999781,1,10000000.00
6026278,0.999781,1,10000000.00
5724271,0.999781,1,10000000.00
6205435,0.999781,1,10000000.00


In [18]:
# Create fraud_amount column — zero for non-fraud rows
fraud_score_transfer['fraud_amount'] = fraud_score_transfer['amount'] * fraud_score_transfer['isFraud']

# Create score buckets
fraud_score_transfer['score_bucket'] = pd.cut(fraud_score_transfer['fraud_score'], 
                                      bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
                                      labels=['0-0.1', '0.1-0.2', '0.2-0.3', '0.3-0.4', '0.4-0.5', 
                                             '0.5-0.6', '0.6-0.7', '0.7-0.8', '0.8-0.9', '0.9-1.0'])

# Build summary dataframe
fraud_by_bucket = fraud_score_transfer.groupby('score_bucket', observed=True).agg(
    fraud_count  = ('isFraud',      'sum'),
    fraud_volume = ('fraud_amount', 'sum')
).reset_index()

# % of total fraud count and volume
fraud_by_bucket['pct_fraud_count']  = (fraud_by_bucket['fraud_count']  / fraud_by_bucket['fraud_count'].sum()  * 100).round(2)
fraud_by_bucket['pct_fraud_volume'] = (fraud_by_bucket['fraud_volume'] / fraud_by_bucket['fraud_volume'].sum() * 100).round(2)

fraud_by_bucket.sort_values(by='score_bucket', inplace=True)
fraud_by_bucket

,score_bucket,fraud_count,fraud_volume,pct_fraud_count,pct_fraud_volume
0,0-0.1,2,2.928374e+06,0.16,0.17
1,0.1-0.2,0,0.000000e+00,0.00,0.00
2,0.2-0.3,0,0.000000e+00,0.00,0.00
3,0.3-0.4,5,3.982882e+05,0.41,0.02
4,0.4-0.5,1,1.035827e+04,0.08,0.00
5,0.5-0.6,4,3.375723e+05,0.33,0.02
6,0.6-0.7,5,2.491112e+04,0.41,0.00
7,0.7-0.8,1,1.027687e+04,0.08,0.00
8,0.8-0.9,1,2.098602e+04,0.08,0.00
9,0.9-1.0,1200,1.708575e+09,98.44,99.78
